In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import BertTokenizer, BertModel

# === Paths ===
BASE_DIR   = Path(r"D:\Shikha\bias_project")
OUTPUT_DIR = BASE_DIR / "outputs" / "embeddings" / "bert"
WORD_LISTS = BASE_DIR / "word_lists" / "caliskan_2017"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Load BERT ===
print("Loading BERT model and tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model     = BertModel.from_pretrained('bert-base-uncased')
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = model.to(device)
print(f"Model loaded on : {device}")
print(f"BERT hidden size: {model.config.hidden_size}")

# === Load WEAT 6 word lists ===
def load_word_list(filepath):
    with open(filepath, 'r') as f:
        return [w.strip().lower() for w in f.readlines() if w.strip()]

career_terms  = load_word_list(WORD_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WORD_LISTS / "family_terms_weat6.txt")
male_names    = load_word_list(WORD_LISTS / "male_names_weat6.txt")
female_names  = load_word_list(WORD_LISTS / "female_names_weat6.txt")

print(f"\nWEAT 6 word lists loaded:")
print(f"Career : {career_terms}")
print(f"Family : {family_terms}")
print(f"Male   : {male_names}")
print(f"Female : {female_names}")

d:\Shikha\bias_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BERT model and tokenizer...


d:\Shikha\bias_project\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhavi\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1129.56it/s]
[transformers] BertModel LOAD REPORT from: bert-base-un

Model loaded on : cpu
BERT hidden size: 768

WEAT 6 word lists loaded:
Career : ['executive', 'management', 'professional', 'corporation', 'salary', 'office', 'business', 'career']
Family : ['home', 'parents', 'children', 'family', 'cousins', 'marriage', 'wedding', 'relatives']
Male   : ['john', 'paul', 'mike', 'kevin', 'steve', 'greg', 'jeff', 'bill']
Female : ['amy', 'joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']


In [2]:
def get_bert_embedding(text, tokenizer, model, device):
    """Get BERT [CLS] token embedding for a sentence"""
    inputs = tokenizer(
        text,
        return_tensors  = 'pt',
        truncation      = True,
        max_length      = 64,
        padding         = True
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Use [CLS] token embedding as sentence representation
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return cls_embedding

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# === SEAT sentence templates ===
# Each word gets embedded in a neutral sentence context
def make_sentence(word):
    return f"This is {word}."

# Get embeddings for all WEAT 6 words
print("Computing BERT embeddings for WEAT 6 word lists...")

career_embs  = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) 
                for w in career_terms}
family_embs  = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) 
                for w in family_terms}
male_embs    = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) 
                for w in male_names}
female_embs  = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) 
                for w in female_names}

print(f"Career embeddings  : {len(career_embs)}")
print(f"Family embeddings  : {len(family_embs)}")
print(f"Male embeddings    : {len(male_embs)}")
print(f"Female embeddings  : {len(female_embs)}")
print(f"Embedding dim      : {list(career_embs.values())[0].shape}")

Computing BERT embeddings for WEAT 6 word lists...
Career embeddings  : 8
Family embeddings  : 8
Male embeddings    : 8
Female embeddings  : 8
Embedding dim      : (768,)


In [3]:
def seat_score(target_A_embs, target_B_embs, attr_C_embs, attr_D_embs):
    """
    SEAT score: measures association of target A vs B with attributes C vs D
    target_A = male names, target_B = female names
    attr_C = career, attr_D = family
    Positive = male names more associated with career (gender bias)
    """
    def mean_cos(target_emb, attr_embs):
        sims = [cosine_sim(target_emb, a) for a in attr_embs.values()]
        return np.mean(sims)

    def association(target_emb, attr_C, attr_D):
        return mean_cos(target_emb, attr_C) - mean_cos(target_emb, attr_D)

    male_assoc   = [association(e, attr_C_embs, attr_D_embs) 
                    for e in target_A_embs.values()]
    female_assoc = [association(e, attr_C_embs, attr_D_embs) 
                    for e in target_B_embs.values()]

    return np.mean(male_assoc) - np.mean(female_assoc)

# === Global SEAT score (no corpus — pure BERT bias) ===
global_seat = seat_score(male_embs, female_embs, career_embs, family_embs)
print(f"=== Global SEAT Score (pretrained BERT) ===")
print(f"Score     : {global_seat:.4f}")
print(f"Direction : {'male→career' if global_seat > 0 else 'female→career'}")
print(f"\nThis reflects bias in BERT pretrained on Wikipedia + BookCorpus.")

=== Global SEAT Score (pretrained BERT) ===
Score     : 0.0063
Direction : male→career

This reflects bias in BERT pretrained on Wikipedia + BookCorpus.


In [4]:
import random

# Load corpus text per state (reuse from GloVe notebook approach)
CORPUS_DIR = BASE_DIR / "corpus"

print("Loading corpora...")
cr_df       = pd.read_csv(CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv")
ca_df       = pd.read_csv(CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles.csv")
gd_df       = pd.read_csv(CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv")
gd_df       = gd_df[gd_df['text'].notna() & (gd_df['text'] != '')]
rd_posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
rd_comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")
print("Done.")

EXCLUDE = {'GU', 'PR', 'VI', 'AS', 'MP'}

# Build per-state sentence lists
state_sentences = {}

def add_sentences(state_abbr, text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return
    # Split into sentences roughly
    sents = [s.strip() for s in text.replace('\n', ' ').split('.') 
             if len(s.strip()) > 20]
    if state_abbr not in state_sentences:
        state_sentences[state_abbr] = []
    state_sentences[state_abbr].extend(sents)

for _, row in cr_df.iterrows():
    add_sentences(str(row['state']), str(row.get('text', '')))
for _, row in ca_df.iterrows():
    add_sentences(str(row['state_abbr']), str(row.get('text', '')))
for _, row in gd_df.iterrows():
    add_sentences(str(row['state']), str(row.get('text', '')))
for _, row in rd_posts.iterrows():
    text = str(row.get('title','')) + '. ' + str(row.get('selftext',''))
    add_sentences(str(row['state_abbr']), text)
for _, row in rd_comments.iterrows():
    add_sentences(str(row['state_abbr']), str(row.get('body','')))

print(f"States with sentences: {len(state_sentences)}")
print(f"Sample — CA sentences: {len(state_sentences.get('CA', []))}")

Loading corpora...
Done.
States with sentences: 52
Sample — CA sentences: 2273


In [5]:
# Per-state SEAT scores
# For each state, we embed WEAT words in context of that state's sentences
# Then compute SEAT score from those contextual embeddings

SENTENCES_PER_WORD = 5  # number of sentences per word to average over
random.seed(42)

def get_contextual_embedding(word, sentences, tokenizer, model, device, n=5):
    """
    Find sentences containing the word, get BERT embedding for each,
    return mean embedding — this captures word meaning in local context
    """
    # Find sentences containing the word
    matching = [s for s in sentences if word.lower() in s.lower()]
    
    if not matching:
        # Fall back to template sentence if word not in corpus
        matching = [f"This is {word}."]
    
    # Sample up to n sentences
    sampled = random.sample(matching, min(n, len(matching)))
    
    embeddings = []
    for sent in sampled:
        emb = get_bert_embedding(sent[:256], tokenizer, model, device)
        embeddings.append(emb)
    
    return np.mean(embeddings, axis=0)

results_bert = []

print(f"{'State':<8} {'Sentences':>10} {'SEAT':>10} {'Direction'}")
print("-" * 50)

for state_abbr in sorted(state_sentences.keys()):
    if state_abbr in EXCLUDE:
        continue

    sentences = state_sentences[state_abbr]
    if len(sentences) < 20:
        print(f"{state_abbr:<8} {len(sentences):>10} {'SKIP':>10} too few sentences")
        continue

    # Get contextual embeddings for all WEAT words
    c_embs = {w: get_contextual_embedding(w, sentences, tokenizer, model, device, SENTENCES_PER_WORD)
              for w in career_terms}
    f_embs = {w: get_contextual_embedding(w, sentences, tokenizer, model, device, SENTENCES_PER_WORD)
              for w in family_terms}
    m_embs = {w: get_contextual_embedding(w, sentences, tokenizer, model, device, SENTENCES_PER_WORD)
              for w in male_names}
    fe_embs = {w: get_contextual_embedding(w, sentences, tokenizer, model, device, SENTENCES_PER_WORD)
               for w in female_names}

    score = seat_score(m_embs, fe_embs, c_embs, f_embs)

    direction = "male→career" if score > 0 else "female→career"
    print(f"{state_abbr:<8} {len(sentences):>10} {score:>10.4f} {direction}")
    results_bert.append({
        'state':      state_abbr,
        'n_sentences': len(sentences),
        'seat_score': round(score, 4),
        'method':     'bert_seat',
        'period':     'mixed_demo'
    })

print(f"\nStates scored : {len(results_bert)}")

State     Sentences       SEAT Direction
--------------------------------------------------
AK             1658     0.0389 male→career
AL              741    -0.0026 female→career
AR              690     0.0155 male→career
AZ             1065     0.0258 male→career
CA             2273     0.0196 male→career
CO              715     0.0048 male→career
CT              841     0.0191 male→career
DE              799     0.0388 male→career
FL             1286     0.0153 male→career
GA             1529     0.0097 male→career
HI             1305     0.0103 male→career
IA             1207     0.0282 male→career
ID              574     0.0117 male→career
IL             2005     0.0026 male→career
IN             2114     0.0182 male→career
KS              554    -0.0066 female→career
KY              830     0.0127 male→career
LA             1100     0.0083 male→career
MA             1329     0.0187 male→career
MD             2086    -0.0089 female→career
ME             1439     0.0057 male→career

In [6]:
# Save BERT SEAT scores
scores_bert_df = pd.DataFrame(results_bert).sort_values('state')

OUTPUT_BERT = OUTPUT_DIR / "seat_scores_bert.csv"
scores_bert_df.to_csv(OUTPUT_BERT, index=False)

print("=== SEAT Scores — BERT (all 50 states) ===\n")
print(f"{'State':<8} {'SEAT Score':>12} {'Bias Direction'}")
print("-" * 40)
for _, row in scores_bert_df.iterrows():
    direction = "male→career" if row['seat_score'] > 0 else "female→career"
    print(f"{row['state']:<8} {row['seat_score']:>12.4f} {direction}")

print(f"\nAll 50 states scored")
print(f"Global BERT baseline  : 0.0063")
print(f"State range           : {scores_bert_df['seat_score'].min():.4f} to {scores_bert_df['seat_score'].max():.4f}")
print(f"Male→career states    : {(scores_bert_df['seat_score'] > 0).sum()}")
print(f"Female→career states  : {(scores_bert_df['seat_score'] < 0).sum()}")
print(f"Saved to              : {OUTPUT_BERT}")

=== SEAT Scores — BERT (all 50 states) ===

State      SEAT Score Bias Direction
----------------------------------------
AK             0.0389 male→career
AL            -0.0026 female→career
AR             0.0155 male→career
AZ             0.0258 male→career
CA             0.0196 male→career
CO             0.0048 male→career
CT             0.0191 male→career
DE             0.0388 male→career
FL             0.0153 male→career
GA             0.0097 male→career
HI             0.0103 male→career
IA             0.0282 male→career
ID             0.0117 male→career
IL             0.0026 male→career
IN             0.0182 male→career
KS            -0.0066 female→career
KY             0.0127 male→career
LA             0.0083 male→career
MA             0.0187 male→career
MD            -0.0089 female→career
ME             0.0057 male→career
MI            -0.0008 female→career
MN            -0.0019 female→career
MO             0.0171 male→career
MS            -0.0018 female→career
MT             0